In [1]:
import re
import csv
import copy
import math
import json
import traceback
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
from scipy.optimize import root
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Patch
from scipy.sparse import csr_matrix, diags, eye
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from dataclasses import dataclass, replace, asdict, is_dataclass
from typing import Any, Dict, Iterable, Iterator, List, Mapping, Optional, Tuple

In [2]:
# ============================================================
#  Parameters
# ============================================================

@dataclass(frozen=True)
class AreaParams:
    """
    Parameters for one brain area.

    N      : number of neurons in this area
    s      : local sparsity, defined by C_i / N_i
             so each neuron in this area receives:
                 C_i = round(s * N)
             local recurrent inputs from the SAME area
    f_exc  : fraction of excitatory neurons in this area
             => N_E = round(f_exc * N), N_I = N - N_E
             => C_E = round(f_exc * C_i), C_I = C_i - C_E
    J_exc  : local excitatory synaptic weight in this area
    g_inh  : local inhibitory strength ratio
             local inhibitory weight = - g_inh * J_exc
    """
    N: int
    s: float
    f_exc: float
    J_exc: float
    g_inh: float


@dataclass(frozen=True)
class MultiAreaParams:
    """
    Parameters for the full multi-area network.

    areas        : tuple of AreaParams, one per area

    s_cross      : shape (A, A)
                   s_cross[src, tgt] is the cross-area sparsity from area src to area tgt
                   Only EXCITATORY neurons in src can project to tgt.
                   For each target neuron in tgt:
                       C_{src->tgt} = round( s_cross[src, tgt] * N_E(src) )

    J_cross      : shape (A, A)
                   J_cross[src, tgt] is the excitatory weight from area src to area tgt

    allow_autapses : whether local self-connections are allowed
    activation     : "tanh" or "threshold_linear"
    phi_max        : used only if activation == "threshold_linear"
    dt, T          : simulation step and total simulation time
    seed           : random seed used to build connectivity
    """
    areas: Tuple[AreaParams, ...]
    s_cross: np.ndarray
    J_cross: np.ndarray
    allow_autapses: bool
    activation: str
    phi_max: float
    gamma: float
    dt: float
    T: float
    seed: int = 0


# ============================================================
#  Activation function
# ============================================================

def phi(x: np.ndarray, p: MultiAreaParams) -> np.ndarray:
    """
    Rate nonlinearity.

    Kept intentionally simple:
    - tanh
    - threshold-linear with optional upper bound phi_max
    """
    if p.activation == "tanh":
        return np.tanh(x) + 1.0

    if p.activation == "threshold_linear":
        y = np.maximum(0.0, (x + p.gamma))
        if np.isfinite(p.phi_max):
            y = np.minimum(p.phi_max, y)
        return y

    raise ValueError(f"Unknown activation = {p.activation}")

# ============================================================
#  Derivative of activation
# ============================================================

def phi_prime(x: np.ndarray, p: MultiAreaParams) -> np.ndarray:
    """
    激活函数导数：
    - tanh: phi'(x) = 1 - tanh(x)^2
    - threshold_linear: x+gamma>0 时导数为 1，否则为 0
      如果设了 phi_max，上饱和区导数为 0
    """
    x = np.asarray(x, dtype=float)

    if p.activation == "tanh":
        y = np.tanh(x)
        return 1.0 - y * y

    if p.activation == "threshold_linear":
        g = ((x + p.gamma) >= 0.0).astype(float)
        if np.isfinite(p.phi_max):
            g = g * ((x + p.gamma) < p.phi_max)
        return g

    raise ValueError(f"Unknown activation = {p.activation}")

# ============================================================
#  Population bookkeeping
#  Population order:
#     (area0,E), (area0,I), (area1,E), (area1,I), ...
# ============================================================

def population_info(p: MultiAreaParams) -> Dict[str, Any]:
    """
    Build all index bookkeeping for the multi-area EI network.

    Returns a dictionary containing:
    - total neuron number
    - slices for each area and each population
    - local degrees C, C_E, C_I for each area
    """
    pops = []
    pop_sizes = []
    pop_slices = {}
    area_slices = {}
    area_stats = []

    start = 0
    for area_id, ap in enumerate(p.areas): # irretating among different areas
        # Number of excitatory / inhibitory neurons in this area
        N_E = int(round(ap.f_exc * ap.N))
        N_I = ap.N - N_E

        if N_E < 0 or N_I < 0:
            raise ValueError(f"Area {area_id}: invalid E/I split")

        # Local fixed in-degree in this area
        C = int(round(ap.s * ap.N))
        C_E = int(round(ap.f_exc * C))
        C_I = C - C_E

        e_slice = slice(start, start + N_E)  # recording the neurons index
        pops.append((area_id, "E"))
        pop_slices[(area_id, "E")] = e_slice
        pop_sizes.append(N_E)
        start += N_E

        i_slice = slice(start, start + N_I)
        pops.append((area_id, "I"))
        pop_slices[(area_id, "I")] = i_slice
        pop_sizes.append(N_I)
        start += N_I

        area_slices[area_id] = slice(e_slice.start, i_slice.stop)

        area_stats.append({
            "area_id": area_id,
            "N": ap.N,
            "N_E": N_E,
            "N_I": N_I,
            "C": C,
            "C_E": C_E,
            "C_I": C_I,
            "J_E": ap.J_exc,
            "J_I": -ap.g_inh * ap.J_exc,
        })

    return {
        "pops": pops,
        "pop_sizes": np.asarray(pop_sizes, dtype=int),
        "N_total": start,
        "pop_slices": pop_slices,
        "area_slices": area_slices,
        "area_stats": area_stats,
    }


# ============================================================
#  Helpers
# ============================================================

def _validate_params(p: MultiAreaParams) -> None:
    """
    Check only the quantities that are essential for this construction.
    """
    A = len(p.areas)

    if p.s_cross.shape != (A, A):
        raise ValueError(f"s_cross must have shape ({A}, {A})")

    if p.J_cross.shape != (A, A):
        raise ValueError(f"J_cross must have shape ({A}, {A})")

    for area_id, ap in enumerate(p.areas):
        if ap.N <= 0:
            raise ValueError(f"Area {area_id}: N must be positive")
        if not (0.0 <= ap.s <= 1.0):
            raise ValueError(f"Area {area_id}: s must be in [0, 1]")
        if not (0.0 <= ap.f_exc <= 1.0):
            raise ValueError(f"Area {area_id}: f_exc must be in [0, 1]")

    if np.any(p.s_cross < 0.0) or np.any(p.s_cross > 1.0):
        raise ValueError("All s_cross entries must be in [0, 1]")


def _candidate_pool(indices: np.ndarray, target_idx: int, allow_autapses: bool) -> np.ndarray:
    """
    Build the candidate presynaptic pool.

    If autapses are forbidden and target neuron belongs to this pool,
    remove itself from the candidate set.
    """
    if allow_autapses or len(indices) == 0:
        return indices

    # If target_idx is inside this pool, remove it; otherwise return as is
    if indices[0] <= target_idx <= indices[-1]:
        return indices[indices != target_idx]
    return indices


# ============================================================
#  Build sparse connectivity matrix
# ============================================================

def build_multi_area_connectivity(p: MultiAreaParams) -> Tuple[csr_matrix, Dict[str, Any]]:
    """
    Build the full sparse recurrent matrix W of the multi-area EI network.

    Construction rule:
    ------------------------------------------------------------
    Local (inside each area i):
      Each target neuron receives exactly:
          C_i   = round(s_i * N_i)
          C_E_i = round(f_i * C_i) excitatory local inputs
          C_I_i = C_i - C_E_i      inhibitory local inputs

      Local weights:
          +J_i               from local excitatory neurons
          -g_i * J_i         from local inhibitory neurons

    Cross-area (src -> tgt):
      Only excitatory neurons in source area src can project.
      Each target neuron in tgt receives exactly:
          C_{src->tgt} = round( s_{src->tgt} * N_E(src) )

      Cross-area weight:
          +J_{src->tgt}
    ------------------------------------------------------------

    Important:
    - This is a random sparse FIXED in-degree construction.
    - Randomness comes from random sampling of presynaptic neurons.
    - Sparsity is controlled by local s_i and cross-area s_{i->j}.
    """
    _validate_params(p)
    rng = np.random.default_rng(p.seed)
    info = population_info(p)

    A = len(p.areas)
    N_total = info["N_total"]
    pop_slices = info["pop_slices"]
    area_stats = info["area_stats"]

    rows = []
    cols = []
    data = []

    # --------------------------------------------------------
    # Loop over target areas
    # --------------------------------------------------------
    for tgt_area in range(A):
        stat_t = area_stats[tgt_area]

        tgt_all = np.arange(
            info["area_slices"][tgt_area].start,
            info["area_slices"][tgt_area].stop
        )
        tgt_exc = np.arange(
            pop_slices[(tgt_area, "E")].start,
            pop_slices[(tgt_area, "E")].stop
        )
        tgt_inh = np.arange(
            pop_slices[(tgt_area, "I")].start,
            pop_slices[(tgt_area, "I")].stop
        )

        C_E_local = stat_t["C_E"]
        C_I_local = stat_t["C_I"]
        w_E_local = stat_t["J_E"]
        w_I_local = stat_t["J_I"]

        # ----------------------------------------------------
        # For each target neuron, draw its presynaptic inputs
        # ----------------------------------------------------
        for i_tgt in tgt_all:
            # ---------- Local E candidates ----------
            exc_pool = _candidate_pool(
                tgt_exc, i_tgt, p.allow_autapses
            )

            # ---------- Local I candidates ----------
            inh_pool = _candidate_pool(
                tgt_inh, i_tgt, p.allow_autapses
            )

            # Check if requested fixed in-degree is feasible
            if C_E_local > len(exc_pool):
                raise ValueError(
                    f"Area {tgt_area}: local C_E={C_E_local} exceeds "
                    f"available local excitatory candidates={len(exc_pool)}"
                )
            if C_I_local > len(inh_pool):
                raise ValueError(
                    f"Area {tgt_area}: local C_I={C_I_local} exceeds "
                    f"available local inhibitory candidates={len(inh_pool)}"
                )

            # ---------- Draw local excitatory inputs ----------
            if C_E_local > 0:
                pre_exc = rng.choice(exc_pool, size=C_E_local, replace=False)
                rows.extend([i_tgt] * C_E_local)
                cols.extend(pre_exc.tolist())
                data.extend([w_E_local] * C_E_local)

            # ---------- Draw local inhibitory inputs ----------
            if C_I_local > 0:
                pre_inh = rng.choice(inh_pool, size=C_I_local, replace=False)
                rows.extend([i_tgt] * C_I_local)
                cols.extend(pre_inh.tolist())
                data.extend([w_I_local] * C_I_local)

            # ------------------------------------------------
            # Cross-area inputs:
            # only EXCITATORY neurons from src can project to tgt
            # ------------------------------------------------
            for src_area in range(A):
                if src_area == tgt_area:
                    continue

                src_exc = np.arange(
                    pop_slices[(src_area, "E")].start,
                    pop_slices[(src_area, "E")].stop
                )

                if len(src_exc) == 0:
                    continue

                s_ij = float(p.s_cross[src_area, tgt_area])
                if s_ij <= 0.0:
                    continue

                C_cross = int(round(s_ij * len(src_exc)))
                if C_cross == 0:
                    continue

                if C_cross > len(src_exc):
                    raise ValueError(
                        f"Cross ({src_area}->{tgt_area}): "
                        f"C_cross={C_cross} exceeds source excitatory pool={len(src_exc)}"
                    )

                pre_cross = rng.choice(src_exc, size=C_cross, replace=False)
                w_cross = float(p.J_cross[src_area, tgt_area])

                rows.extend([i_tgt] * C_cross)
                cols.extend(pre_cross.tolist())
                data.extend([w_cross] * C_cross)

    W = csr_matrix(
        (
            np.asarray(data, dtype=float),
            (np.asarray(rows, dtype=int), np.asarray(cols, dtype=int))
        ),
        shape=(N_total, N_total),
    )

    meta = {
        "N_total": N_total,
        "pops": info["pops"],
        "pop_sizes": info["pop_sizes"],
        "pop_slices": pop_slices,
        "area_slices": info["area_slices"],
        "area_stats": area_stats,
        "nnz": W.nnz,
    }
    return W, meta

# ============================================================
#  Lift population values -> neuron values
# ============================================================

def lift_population_values_to_neurons(values: np.ndarray, p: MultiAreaParams) -> np.ndarray:
    """
    Extend population vector length 2A into length N_total neuron vector.

    example:
      values = [x_(area0,E), x_(area0,I), x_(area1,E), x_(area1,I), ...]
      like for current I
    """
    info = population_info(p)
    pops = info["pops"]
    pop_slices = info["pop_slices"]
    N_total = info["N_total"]

    out = np.empty(N_total, dtype=float)
    for k, (area_id, cell_type) in enumerate(pops):
        out[pop_slices[(area_id, cell_type)]] = values[k]
    return out

# ============================================================
#  Full-network fixed point solver
#  Solve directly:
#       x* = W @ phi(x*)
#  i.e.
#       F(x) = -x + W @ phi(x) = 0
# ============================================================

def solve_full_fixed_point(
    W: csr_matrix,
    p: MultiAreaParams,
    x0_init: Optional[np.ndarray] = None,
    dt_fp: Optional[float] = None,
    tol: float = 1e-5,
    max_iter: int = 200000,
    verbose: bool = False,
    max_abs_x: float = 1e6,
    adaptive: bool = True,
) -> np.ndarray:
    """

    Solve:
        dx/dt = -x + W @ phi(x)
    and stop when the drift
        F(x) = -x + W @ phi(x)
    is sufficiently small
    """
    N = W.shape[0]

    if x0_init is None:
        x = np.zeros(N, dtype=float)
    else:
        x = np.asarray(x0_init, dtype=float).copy()
        if x.shape != (N,):
            raise ValueError(f"x0_init must have shape ({N},)")

    if dt_fp is None:
        # use same step size as simulation
        dt_fp = p.dt

    if dt_fp <= 0.0:
        raise ValueError("dt_fp must be positive")

    prev_residual = np.inf

    for k in range(max_iter):
        r = phi(x, p)
        F = -x + W @ r

        # divergence check (drift)
        residual = np.max(np.abs(F))

        if verbose and (k < 10 or k % 1000 == 0):
            print(f"[iter {k:6d}] dt_fp = {dt_fp:.3e}, residual = {residual:.3e}, "
                  f"max|x| = {np.max(np.abs(x)):.3e}")

        if not np.all(np.isfinite(F)) or not np.all(np.isfinite(x)):
            if adaptive and dt_fp > 1e-8:
                dt_fp *= 0.5
                if verbose:
                    print(f"Non-finite encountered, reducing dt_fp to {dt_fp:.3e} and retrying.")
                # stable if start from 0, but could also continue from last step
                x = np.zeros(N, dtype=float) if x0_init is None else np.asarray(x0_init, dtype=float).copy()
                prev_residual = np.inf
                continue
            raise RuntimeError("Non-finite values encountered during fixed-point solve.")

        if residual < tol:
            if verbose:
                print(f"Converged at iter {k}: residual = {residual:.3e}")
            return x

        # Euler update
        x_new = x + dt_fp * F

        if not np.all(np.isfinite(x_new)) or np.max(np.abs(x_new)) > max_abs_x:
            if adaptive and dt_fp > 1e-8:
                dt_fp *= 0.5
                if verbose:
                    print(f"State exploded, reducing dt_fp to {dt_fp:.3e} and retrying.")
                continue
            raise RuntimeError(
                f"State diverged during fixed-point solve. "
                f"max|x_new|={np.max(np.abs(x_new)):.3e}"
            )

        # Automatically decrease step size if residual get worse
        if adaptive and residual > prev_residual * 1.2 and dt_fp > 1e-8:
            dt_fp *= 0.5
            if verbose:
                print(f"Residual increased, reducing dt_fp to {dt_fp:.3e}")
            continue

        x = x_new
        prev_residual = residual

    raise RuntimeError(
        f"Full fixed point solver did not converge within max_iter={max_iter}. "
        f"Last residual={residual:.3e}, max|x|={np.max(np.abs(x)):.3e}, dt_fp={dt_fp:.3e}"
    )

In [3]:
# ============================================================
#  Mean connectivity S (dense), numerical construction
# ============================================================

def build_mean_connectivity_dense(p: MultiAreaParams) -> np.ndarray:
    """
    Build the mean connectivity matrix S = E[W] numerically
    from the same block means used in your note.

    Convention:
      W[i, j] = connection from neuron j to neuron i

    We use the same simplified mean structure as in the note:
    - within one area:
        source E projects with mean  J_exc * C_E / N_E
        source I projects with mean -g*J_exc * C_I / N_I
      onto both E and I targets in the same area
    - across areas:
        only source E projects, with mean J_cross * C_cross / N_E(src)
      onto all neurons in the target area
    """
    info = population_info(p)
    pop_slices = info["pop_slices"]
    area_slices = info["area_slices"]
    area_stats = info["area_stats"]
    N_total = info["N_total"]
    A = len(p.areas)

    S = np.zeros((N_total, N_total), dtype=float)

    for tgt in range(A):
        tgt_all = area_slices[tgt]

        for src in range(A):
            src_E = pop_slices[(src, "E")]
            src_I = pop_slices[(src, "I")]
            stat_s = area_stats[src]

            N_E_src = stat_s["N_E"]
            N_I_src = stat_s["N_I"]

            if src == tgt:
                # local E source mean
                mu_E = 0.0 if N_E_src == 0 else stat_s["J_E"] * stat_s["C_E"] / N_E_src
                # local I source mean
                mu_I = 0.0 if N_I_src == 0 else stat_s["J_I"] * stat_s["C_I"] / N_I_src

                S[tgt_all, src_E] = mu_E
                S[tgt_all, src_I] = mu_I

            else:
                # cross-area: only source E projects
                q = float(p.s_cross[src, tgt])
                C_cross = int(round(q * N_E_src))
                mu_cross = 0.0 if N_E_src == 0 else float(p.J_cross[src, tgt]) * C_cross / N_E_src

                S[tgt_all, src_E] = mu_cross
                S[tgt_all, src_I] = 0.0

    return S


# ============================================================
#  Fixed-point summaries by area
# ============================================================

def summarize_fixed_point_by_area(x_star: np.ndarray, p: MultiAreaParams) -> dict:
    """
    Return per-area mean current and mean firing rate.
    """
    info = population_info(p)
    area_slices = info["area_slices"]
    A = len(p.areas)

    out = {}
    for area_id in range(A):
        sl = area_slices[area_id]
        x_area = x_star[sl]
        r_area = phi(x_area, p)

        out[f"x0_area{area_id}_mean"] = float(np.mean(x_area))
        out[f"rate_area{area_id}_mean"] = float(np.mean(r_area))
    return out


# ============================================================
#  Numerical diagnostics for one realization
# ============================================================
def compute_realization_diagnostics(
    W: csr_matrix,
    p: MultiAreaParams,
    solve_fixed_point: bool = True,
) -> dict:
    """
    Compute numerical diagnostics for one realization.

    Logged quantities:
    - x0 mean over all neurons
    - max component of x0
    - max real part of eigenvalues of R = W - S
    - max real part of eigenvalues of S
    - optional per-area means
    """
    W_dense = W.toarray()
    S_dense = build_mean_connectivity_dense(p)
    R_dense = W_dense - S_dense

    # ----- spectral quantities -----
    eig_R = np.linalg.eigvals(R_dense)
    eig_S = np.linalg.eigvals(S_dense)
    eig_S_nonzero = eig_S[np.abs(eig_S) > 1e-6]

    S_max_real_nonzero_eig = (
        float(np.max(eig_S_nonzero.real))
        if eig_S_nonzero.size > 0
        else 0.0
    )
    
    result = {
        "R_max_real_eig": float(np.max(eig_R.real)),
        "eig_S": S_max_real_nonzero_eig,
        "non_zero_W": int(W.nnz),
    }


    # ----- fixed point -----
    if solve_fixed_point:
        try:
            x_star = solve_full_fixed_point(
                W=W,
                p=p,
                x0_init=None,
                dt_fp=p.dt,
                tol=1e-5,
                max_iter=200000,
                verbose=False,
                adaptive=True,
            )

            result["fixed_point_success"] = 1
            result["x0_mean"] = float(np.mean(x_star))
            result["x0_max"] = float(np.max(x_star))
            result["x0_min"] = float(np.min(x_star))

            # per-area summary
            result.update(summarize_fixed_point_by_area(x_star, p))

        except Exception as e:
            result["fixed_point_success"] = 0
            result["x0_mean"] = np.inf
            result["x0_max"] = np.inf
            result["x0_min"] = np.inf
            result["fixed_point_error"] = str(e)
    
            for area_id in range(len(p.areas)):
                result[f"x0_area{area_id}_mean"] = np.inf
                result[f"rate_area{area_id}_mean"] = np.inf

    return result

In [4]:
def to_jsonable(obj):
    """
    Convert object to JSON-safe form.
    Non-finite floats are stored as strings:
        np.inf   -> "__INF__"
        -np.inf  -> "__-INF__"
        np.nan   -> "__NAN__"
    """
    if is_dataclass(obj):
        return to_jsonable(asdict(obj))

    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())

    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)

    if isinstance(obj, (np.floating, float)):
        x = float(obj)
        if np.isnan(x):
            return "__NAN__"
        if np.isposinf(x):
            return "__INF__"
        if np.isneginf(x):
            return "__-INF__"
        return x

    if isinstance(obj, (np.integer, int)):
        return int(obj)

    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]

    return obj

def flatten_params(p: MultiAreaParams) -> Dict[str, Any]:
    """
    Flatten all parameters in MultiAreaParams into a flat dict for logging.
    Example keys:
      areas[0].N
      areas[0].s
      s_cross[0,1]
      J_cross[1,0]
      dt
      T
      seed
    """
    out: Dict[str, Any] = {}

    for i, ap in enumerate(p.areas):
        out[f"areas[{i}].N"] = int(ap.N)
        out[f"areas[{i}].s"] = float(ap.s)
        out[f"areas[{i}].f_exc"] = float(ap.f_exc)
        out[f"areas[{i}].J_exc"] = float(ap.J_exc)
        out[f"areas[{i}].g_inh"] = float(ap.g_inh)

    A = len(p.areas)
    for i in range(A):
        for j in range(A):
            out[f"s_cross[{i},{j}]"] = float(p.s_cross[i, j])
            out[f"J_cross[{i},{j}]"] = float(p.J_cross[i, j])

    out["allow_autapses"] = bool(p.allow_autapses)
    out["activation"] = p.activation
    out["phi_max"] = float(p.phi_max) if np.isfinite(p.phi_max) else np.inf
    out["gamma"] = float(p.gamma)
    out["dt"] = float(p.dt)
    out["T"] = float(p.T)
    out["seed"] = int(p.seed)

    return out

def apply_updates_to_params(base_p: MultiAreaParams, updates: Mapping[str, Any]) -> MultiAreaParams:
    """
    Apply updates like:
        "areas[0].s"   -> 0.08
        "areas[1].J_exc" -> 0.04
        "s_cross[0,1]" -> 0.02
        "J_cross[1,0]" -> 0.05
        "gamma"        -> 0.3

    Returns a NEW MultiAreaParams.
    """
    # Start from copies
    areas = list(base_p.areas)
    s_cross = np.array(base_p.s_cross, copy=True)
    J_cross = np.array(base_p.J_cross, copy=True)

    # Scalar fields in MultiAreaParams that may be updated directly
    top_fields = {
        "allow_autapses": base_p.allow_autapses,
        "activation": base_p.activation,
        "phi_max": base_p.phi_max,
        "gamma": base_p.gamma,
        "dt": base_p.dt,
        "T": base_p.T,
        "seed": base_p.seed,
    }

    for key, value in updates.items():
        key = key.replace(" ", "")

        if key.startswith("areas["):
            # e.g. areas[0].s
            left, field = key.split("].")
            area_idx = int(left[len("areas["):])
            old_area = areas[area_idx]
            areas[area_idx] = replace(old_area, **{field: value})

        elif key.startswith("s_cross["):
            # e.g. s_cross[0,1]
            inside = key[len("s_cross["):-1]
            i, j = map(int, inside.split(","))
            s_cross[i, j] = value

        elif key.startswith("J_cross["):
            # e.g. J_cross[1,0]
            inside = key[len("J_cross["):-1]
            i, j = map(int, inside.split(","))
            J_cross[i, j] = value

        elif key in top_fields:
            top_fields[key] = value

        else:
            raise KeyError(f"Unsupported parameter path: {key}")

    new_p = MultiAreaParams(
        areas=tuple(areas),
        s_cross=s_cross,
        J_cross=J_cross,
        allow_autapses=top_fields["allow_autapses"],
        activation=top_fields["activation"],
        phi_max=top_fields["phi_max"],
        gamma=top_fields["gamma"],
        dt=top_fields["dt"],
        T=top_fields["T"],
        seed=top_fields["seed"],
    )
    return new_p

In [5]:
def aggregate_diagnostics(diags_list: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregate a list of per-realization diagnostics dicts.

    For each numeric key found in any realization, compute the mean and
    std across realizations using only finite values. Also report how
    many realizations contributed a finite value for that key.

    Non-numeric keys (e.g. error messages) are skipped.
    """
    agg: Dict[str, Any] = {"n_realizations": len(diags_list)}

    # Collect every numeric key that appears in any realization
    keys = set()
    for d in diags_list:
        for k, v in d.items():
            if isinstance(v, bool):
                continue
            if isinstance(v, (int, float, np.integer, np.floating)):
                keys.add(k)

    for k in sorted(keys):
        vals = np.array([d.get(k, np.nan) for d in diags_list], dtype=float)
        finite = vals[np.isfinite(vals)]
        if finite.size > 0:
            agg[f"{k}_mean"] = float(np.mean(finite))
            agg[f"{k}_std"]  = float(np.std(finite, ddof=0))
        else:
            agg[f"{k}_mean"] = np.inf
            agg[f"{k}_std"]  = np.inf
        agg[f"{k}_n_finite"] = int(finite.size)

    return agg


def run_parameter_scan(
    base_p: MultiAreaParams,
    scan_points: Iterable[Mapping[str, Any]],
    output_file: str | Path,
    solve_fixed_point: bool = True,
    flush_every: int = 1,
    stop_on_error: bool = False,
    n_realizations: int = 1,
    keep_realizations: bool = False,
) -> None:
    """
    Run a parameter scan and write one JSON record per scan point.

    For each scan point we build `n_realizations` INDEPENDENT random
    connectivity matrices (using seeds base_p.seed, base_p.seed + 1, ...,
    base_p.seed + n_realizations - 1) and aggregate the numerical
    diagnostics across these realizations. This gives proper ensemble
    statistics (mean and std) at each parameter point instead of relying
    on a single random matrix.

    Each output line contains:
    - scan_index
    - scanned_updates
    - n_realizations
    - flattened parameters (recorded with the base seed)
    - meta (N_total, mean nnz across realizations)
    - aggregated diagnostics: for every numeric diagnostic key `k`,
      we store `k_mean`, `k_std`, `k_n_finite`
    - if keep_realizations=True, also the raw per-realization diagnostics
    - status / error info
    """
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with output_file.open("w", encoding="utf-8") as f:
        for scan_idx, updates in enumerate(scan_points):
            record: Dict[str, Any] = {
                "scan_index": scan_idx,
                "scanned_updates": dict(updates),
                "n_realizations": int(n_realizations),
            }

            try:
                # 1) apply parameter updates (keeps base seed for now)
                p_base_run = apply_updates_to_params(base_p, updates)

                # 2) flatten parameters for logging
                record["params"] = flatten_params(p_base_run)

                base_seed = int(p_base_run.seed)
                realization_diags: List[Dict[str, Any]] = []
                realization_meta:  List[Dict[str, Any]] = []

                # 3) loop over independent random realizations
                for k in range(n_realizations):
                    p_run = apply_updates_to_params(
                        p_base_run, {"seed": base_seed + k}
                    )

                    W, meta = build_multi_area_connectivity(p_run)

                    diag = compute_realization_diagnostics(
                        W=W,
                        p=p_run,
                        solve_fixed_point=solve_fixed_point,
                    )
                    diag["seed"] = base_seed + k

                    realization_diags.append(diag)
                    realization_meta.append({
                        "seed": base_seed + k,
                        "N_total": int(meta["N_total"]),
                        "n_non_zero": int(meta["nnz"]),
                    })

                # 4) aggregate across realizations
                record["meta"] = {
                    "N_total": realization_meta[0]["N_total"],
                    "n_non_zero_mean": float(
                        np.mean([m["n_non_zero"] for m in realization_meta])
                    ),
                    "n_non_zero_std": float(
                        np.std([m["n_non_zero"] for m in realization_meta], ddof=0)
                    ),
                }
                record["diagnostics"] = aggregate_diagnostics(realization_diags)
                if keep_realizations:
                    record["realizations"] = realization_diags
                record["status"] = "ok"

            except Exception as e:
                record["status"] = "error"
                record["error_type"] = type(e).__name__
                record["error_message"] = str(e)
                record["traceback"] = traceback.format_exc()

                if stop_on_error:
                    f.write(json.dumps(to_jsonable(record), ensure_ascii=False) + "\n")
                    f.flush()
                    raise

            f.write(json.dumps(to_jsonable(record), ensure_ascii=False) + "\n")

            if (scan_idx + 1) % flush_every == 0:
                f.flush()

In [6]:
area0 = AreaParams(N=2000, s=0.05, f_exc=0.8, J_exc=0.035, g_inh=4.5)
area1 = AreaParams(N=2000, s=0.05, f_exc=0.8, J_exc=0.035, g_inh=4.5)

# s_cross_fixed = np.zeros((2, 2))
# s_cross_fixed[0, 1] = 0.025
# s_cross_fixed[1, 0] = 0.025

# J_cross_fixed = np.zeros((2, 2))
# J_cross_fixed[0, 1] = 0.01
# J_cross_fixed[1, 0] = 0.01
'''
J_cross[src, tgt]
'''

base_p = MultiAreaParams(
    areas=(area0, area1),
    s_cross=np.zeros((2, 2)),
    #s_cross=s_cross_fixed,
    J_cross=np.zeros((2, 2)),
    #J_cross=J_cross_fixed,
    allow_autapses=False,
    activation="threshold_linear",
    phi_max=np.inf,
    gamma=0.5,
    dt=0.005,
    T=60.0,
    seed=0,
)

scan_points = (
    {
        "J_cross[1,0]": J0,
        "J_cross[0,1]": J0,
        "s_cross[1,0]": s0,
        "s_cross[0,1]": s0,
    }
    for J0, s0 in product(
        np.linspace(0.0, 0.05, 11),
        np.linspace(0.0, 0.05, 11)
    )
)

run_parameter_scan(
    base_p=base_p,
    scan_points=scan_points,
    output_file="scan_J_cross_s_cross_J_within035.jsonl",
    solve_fixed_point=True,
    flush_every=1,
    n_realizations=8,        # 每个参数点用 10 个不同随机矩阵
    keep_realizations=False,  # True 时还会保存每次 realization 的原始诊断

)

In [7]:
area0 = AreaParams(N=2000, s=0.05, f_exc=0.8, J_exc=0.045, g_inh=4.5)
area1 = AreaParams(N=2000, s=0.05, f_exc=0.8, J_exc=0.045, g_inh=4.5)

# s_cross_fixed = np.zeros((2, 2))
# s_cross_fixed[0, 1] = 0.025
# s_cross_fixed[1, 0] = 0.025

# J_cross_fixed = np.zeros((2, 2))
# J_cross_fixed[0, 1] = 0.01
# J_cross_fixed[1, 0] = 0.01
'''
J_cross[src, tgt]
'''

base_p = MultiAreaParams(
    areas=(area0, area1),
    s_cross=np.zeros((2, 2)),
    #s_cross=s_cross_fixed,
    J_cross=np.zeros((2, 2)),
    #J_cross=J_cross_fixed,
    allow_autapses=False,
    activation="threshold_linear",
    phi_max=np.inf,
    gamma=0.5,
    dt=0.005,
    T=60.0,
    seed=0,
)

scan_points = (
    {
        "J_cross[1,0]": J0,
        "J_cross[0,1]": J0,
        "s_cross[1,0]": s0,
        "s_cross[0,1]": s0,
    }
    for J0, s0 in product(
        np.linspace(0.0, 0.1, 17),
        np.linspace(0.0, 0.1, 17)
    )
)

run_parameter_scan(
    base_p=base_p,
    scan_points=scan_points,
    output_file="scan_J_cross_s_cross_J_within045.jsonl",
    solve_fixed_point=True,
    flush_every=1,
    n_realizations=8,        # 每个参数点用 10 个不同随机矩阵
    keep_realizations=False,  # True 时还会保存每次 realization 的原始诊断

)

In [11]:
area0 = AreaParams(N=1000, s=0.1, f_exc=0.8, J_exc=0.035, g_inh=4.5)
area1 = AreaParams(N=1000, s=0.1, f_exc=0.8, J_exc=0.035, g_inh=4.5)

s_cross_fixed = np.zeros((2, 2))
s_cross_fixed[0, 1] = 0.05
s_cross_fixed[1, 0] = 0.05

# J_cross_fixed = np.zeros((2, 2))
# J_cross_fixed[0, 1] = 0.01
# J_cross_fixed[1, 0] = 0.01
'''
J_cross[src, tgt]
'''

base_p = MultiAreaParams(
    areas=(area0, area1),
    #s_cross=np.zeros((2, 2)),
    s_cross=s_cross_fixed,
    J_cross=np.zeros((2, 2)),
    #J_cross=J_cross_fixed,
    allow_autapses=False,
    activation="threshold_linear",
    phi_max=np.inf,
    gamma=0.5,
    dt=0.005,
    T=60.0,
    seed=0,
)

scan_points = (
    {
        "J_cross[1,0]": J0,
        "J_cross[0,1]": J0,
        "areas[0].J_exc": Jw,
        "areas[1].J_exc": Jw,
    }
    for J0, Jw in product(
        np.linspace(0.0, 0.05, 11),
        np.linspace(0.0, 0.08, 16)
    )
)

run_parameter_scan(
    base_p=base_p,
    scan_points=scan_points,
    output_file="scan_J_cross_J_within.jsonl",
    solve_fixed_point=True,
    flush_every=1,
    n_realizations=10,        # 每个参数点用 10 个不同随机矩阵
    keep_realizations=False,  # True 时还会保存每次 realization 的原始诊断

)

In [9]:
'''
scan_points = (
    {
        "areas[0].s": s0,
        "areas[1].s": s1,
        "areas[0].J_exc": J0,
        "areas[1].J_exc": J1,
    }
    for s0, s1, J0, J1 in product(
        np.linspace(0.03, 0.08, 6),
        np.linspace(0.03, 0.08, 6),
        np.linspace(0.02, 0.05, 7),
        np.linspace(0.02, 0.05, 7),
    )
)
'''

'\nscan_points = (\n    {\n        "areas[0].s": s0,\n        "areas[1].s": s1,\n        "areas[0].J_exc": J0,\n        "areas[1].J_exc": J1,\n    }\n    for s0, s1, J0, J1 in product(\n        np.linspace(0.03, 0.08, 6),\n        np.linspace(0.03, 0.08, 6),\n        np.linspace(0.02, 0.05, 7),\n        np.linspace(0.02, 0.05, 7),\n    )\n)\n'